In [1]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42
print("lightgbm version:", lgb.__version__)

lightgbm version: 4.7.0


In [2]:
data_path = Path("../data/processed/clean_metro_housing.parquet")
df = pd.read_parquet(data_path)

print("Shape:", df.shape)
df.head()

Shape: (27163, 9)


,City,Locality,Location,BHK,Area_SqFt,Resale,Amenity_Score,Amenity_Info_Available,Price_Lakhs
0,Bangalore,Bangalore | Jp Nagar Phase 1,Jp Nagar Phase 1,4,3340,0,14.0,1,300.000000
1,Bangalore,Bangalore | Dasarahalli On Tumkur Road,Dasarahalli On Tumkur Road,2,1045,0,15.0,1,78.879997
2,Bangalore,Bangalore | Kannur On Thanisandra Main Road,Kannur On Thanisandra Main Road,2,1179,0,9.0,1,48.660000
3,Bangalore,Bangalore | Doddanekundi,Doddanekundi,3,1675,0,3.0,1,83.580002
4,Bangalore,Bangalore | Kengeri,Kengeri,3,1670,0,16.0,1,68.449997


In [3]:
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())
print()
print("Memory (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))
print("Different localities:", df["Locality"].nunique())

City                      category
Locality                  category
Location                  category
BHK                           int8
Area_SqFt                    int32
Resale                        int8
Amenity_Score              float32
Amenity_Info_Available        int8
Price_Lakhs                float32
dtype: object

Missing values per column:
City                          0
Locality                      0
Location                      0
BHK                           0
Area_SqFt                     0
Resale                        0
Amenity_Score             18972
Amenity_Info_Available        0
Price_Lakhs                   0
dtype: int64

Memory (MB): 0.6
Different localities: 1700


# Target, features & train/test split

**Define features and the target**

In [4]:
feature_columns = ["City", "Locality", "BHK", "Area_SqFt",
                   "Resale", "Amenity_Score", "Amenity_Info_Available"]

df["Log_Price"] = np.log1p(df["Price_Lakhs"])   # log(1 + price in lakhs)

print("Features:", feature_columns)
print(df[["Price_Lakhs", "Log_Price"]].describe().round(2))

Features: ['City', 'Locality', 'BHK', 'Area_SqFt', 'Resale', 'Amenity_Score', 'Amenity_Info_Available']
       Price_Lakhs  Log_Price
count     27163.00   27163.00
mean         96.43       4.29
std          98.76       0.71
min          20.00       3.04
25%          42.00       3.76
50%          67.27       4.22
75%         113.00       4.74
max        2027.00       7.61


**Split, stratified by City**

In [5]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["City"]
)

print("Train rows:", len(train_df), "| Test rows:", len(test_df))

share_table = pd.DataFrame({
    "train_%": (train_df["City"].value_counts(normalize=True) * 100).round(1),
    "test_%": (test_df["City"].value_counts(normalize=True) * 100).round(1),
    "test_rows": test_df["City"].value_counts(),
})
print(share_table)

Train rows: 21730 | Test rows: 5433
           train_%  test_%  test_rows
City                                 
Mumbai        23.4    23.4       1273
Kolkata       20.9    20.9       1137
Bangalore     19.5    19.5       1057
Chennai       15.1    15.1        823
Delhi         13.7    13.7        746
Hyderabad      7.3     7.3        397


**How many test localities has the model never seen?**

In [6]:
seen_in_train = test_df["Locality"].isin(train_df["Locality"].unique())
unseen_rows = test_df[~seen_in_train]

print("Test rows in localities never seen in train:", len(unseen_rows),
      "| share:", round(len(unseen_rows) / len(test_df) * 100, 1), "%")
print(unseen_rows["City"].value_counts())

# the category lists should be identical on both sides
same_categories = list(train_df["Locality"].cat.categories) == list(test_df["Locality"].cat.categories)
print("Locality category list identical in train and test:", same_categories)

Test rows in localities never seen in train: 136 | share: 2.5 %
City
Delhi        35
Hyderabad    30
Kolkata      26
Mumbai       23
Bangalore    15
Chennai       7
Name: count, dtype: int64
Locality category list identical in train and test: True


# Error measure and baselines

**error report function**

In [7]:
def report_errors(actual_lakhs, predicted_lakhs, cities):
    # make predicted a Series with the same row labels as actual
    predicted_lakhs = pd.Series(predicted_lakhs, index=actual_lakhs.index)
    cities = cities.astype(str)

    rows = []
    for city_name in sorted(cities.unique()):
        in_city = (cities == city_name)
        lakh_error = (predicted_lakhs[in_city] - actual_lakhs[in_city]).abs()
        percent_error = lakh_error / actual_lakhs[in_city] * 100
        rows.append([city_name, in_city.sum(), percent_error.median(), lakh_error.mean()])

    # one extra row for all cities together
    all_lakh_error = (predicted_lakhs - actual_lakhs).abs()
    all_percent_error = all_lakh_error / actual_lakhs * 100
    rows.append(["ALL", len(actual_lakhs), all_percent_error.median(), all_lakh_error.mean()])

    table = pd.DataFrame(rows, columns=["City", "rows", "median_pct_error", "mae_lakhs"])
    return table.set_index("City").round(1)

**Baseline 1, city median price per sq ft times area**

In [8]:
# price per sq ft in rupees, from TRAIN rows only
train_psf = train_df["Price_Lakhs"] * 100000 / train_df["Area_SqFt"]

# the typical (median) value for each city, as a dictionary: city -> price per sq ft
city_median_psf = train_psf.groupby(train_df["City"], observed=True).median()
city_psf_dict = city_median_psf.astype(float).to_dict()
print(city_median_psf.round(0))

# give each test flat its city's value, then multiply by area and convert to lakhs
test_city_psf = test_df["City"].astype(str).map(city_psf_dict)
baseline1_pred = test_city_psf * test_df["Area_SqFt"] / 100000

baseline1_report = report_errors(test_df["Price_Lakhs"], baseline1_pred, test_df["City"])
print(baseline1_report)

City
Bangalore     5299.0
Chennai       5417.0
Delhi         6731.0
Hyderabad     5100.0
Kolkata       4292.0
Mumbai       10625.0
dtype: float64
           rows  median_pct_error  mae_lakhs
City                                        
Bangalore  1057              30.0       42.1
Chennai     823              23.5       32.0
Delhi       746              38.1       48.2
Hyderabad   397              22.0       32.4
Kolkata    1137              37.6       25.3
Mumbai     1273              46.9       67.0
ALL        5433              35.0       43.0


**Baseline 2, locality median price per sq ft times area**

In [9]:
test_locality = test_df["Locality"].astype(str)

# locality median price per sq ft (train only), and how many train rows each locality has
locality_median_psf = train_psf.groupby(train_df["Locality"], observed=True).median()
locality_psf_dict = locality_median_psf.astype(float).to_dict()
locality_count_dict = train_df["Locality"].value_counts().to_dict()

test_locality_psf = test_locality.map(locality_psf_dict)                   # NaN if unseen
test_locality_count = test_locality.map(locality_count_dict).fillna(0)

# use the locality value where it has 5+ train rows, otherwise the city value
enough_data = test_locality_count >= 5
chosen_psf = pd.Series(np.where(enough_data, test_locality_psf, test_city_psf),
                       index=test_df.index)
baseline2_pred = chosen_psf * test_df["Area_SqFt"] / 100000

print("Test rows using the locality median:", enough_data.sum(), "of", len(test_df))
baseline2_report = report_errors(test_df["Price_Lakhs"], baseline2_pred, test_df["City"])
print(baseline2_report)

Test rows using the locality median: 4985 of 5433
           rows  median_pct_error  mae_lakhs
City                                        
Bangalore  1057              24.7       35.8
Chennai     823              16.6       25.9
Delhi       746              17.7       33.8
Hyderabad   397              11.8       18.7
Kolkata    1137              38.0       25.5
Mumbai     1273              35.9       57.4
ALL        5433              25.9       35.7


**both baselines side by side**

In [10]:
comparison = pd.DataFrame({
    "baseline1_pct": baseline1_report["median_pct_error"],
    "baseline2_pct": baseline2_report["median_pct_error"],
    "baseline1_mae": baseline1_report["mae_lakhs"],
    "baseline2_mae": baseline2_report["mae_lakhs"],
})
print(comparison)

           baseline1_pct  baseline2_pct  baseline1_mae  baseline2_mae
City                                                                 
Bangalore           30.0           24.7           42.1           35.8
Chennai             23.5           16.6           32.0           25.9
Delhi               38.1           17.7           48.2           33.8
Hyderabad           22.0           11.8           32.4           18.7
Kolkata             37.6           38.0           25.3           25.5
Mumbai              46.9           35.9           67.0           57.4
ALL                 35.0           25.9           43.0           35.7


# first LightGBM model

**Carve a validation set out of the training rows**

In [11]:
fit_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=RANDOM_STATE, stratify=train_df["City"]
)

print("Fit rows:", len(fit_df), "| Validation rows:", len(val_df))
print(val_df["City"].value_counts())

Fit rows: 17384 | Validation rows: 4346
City
Mumbai       1018
Kolkata       910
Bangalore     845
Chennai       658
Delhi         597
Hyderabad     318
Name: count, dtype: int64


**Baseline on the validation rows (replacement)**

In [12]:
def locality_baseline(fit_rows, target_rows, min_rows=5):
    # price per sq ft in rupees, from the fit rows only
    fit_psf = fit_rows["Price_Lakhs"] * 100000 / fit_rows["Area_SqFt"]

    city_psf = fit_psf.groupby(fit_rows["City"], observed=True).median().astype(float).to_dict()
    locality_psf = fit_psf.groupby(fit_rows["Locality"], observed=True).median().astype(float).to_dict()
    locality_count = fit_rows["Locality"].value_counts().to_dict()

    target_city_psf = target_rows["City"].astype(str).map(city_psf)
    target_locality = target_rows["Locality"].astype(str)
    target_locality_psf = target_locality.map(locality_psf)

    # use the locality value when it has enough fit rows, otherwise the city value
    enough_rows = target_locality.map(locality_count).fillna(0) >= min_rows
    chosen_psf = np.where(enough_rows, target_locality_psf, target_city_psf)

    return pd.Series(chosen_psf, index=target_rows.index) * target_rows["Area_SqFt"] / 100000


val_baseline_pred = locality_baseline(fit_df, val_df)
val_baseline_report = report_errors(val_df["Price_Lakhs"], val_baseline_pred, val_df["City"])
print(val_baseline_report)

           rows  median_pct_error  mae_lakhs
City                                        
Bangalore   845              22.3       31.5
Chennai     658              19.4       29.2
Delhi       597              19.2       33.9
Hyderabad   318              12.0       18.7
Kolkata     910              39.1       26.4
Mumbai     1018              37.5       58.2
ALL        4346              26.0       35.7


**Training LightGBM with early stopping**

In [13]:
X_fit = fit_df[feature_columns]
y_fit = fit_df["Log_Price"]
X_val = val_df[feature_columns]
y_val = val_df["Log_Price"]

model = lgb.LGBMRegressor(
    n_estimators=3000,       # maximum trees, early stopping will end it sooner
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    verbose=-1,
)

model.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False),
               lgb.log_evaluation(period=0)],
)

print("Trees actually used:", model.best_iteration_)

Trees actually used: 202


**Comparision with the baseline and check for overfitting**

In [14]:
val_pred_lakhs = np.expm1(model.predict(X_val))
lgbm_report = report_errors(val_df["Price_Lakhs"], val_pred_lakhs, val_df["City"])

comparison = pd.DataFrame({
    "baseline_pct": val_baseline_report["median_pct_error"],
    "lgbm_pct": lgbm_report["median_pct_error"],
    "baseline_mae": val_baseline_report["mae_lakhs"],
    "lgbm_mae": lgbm_report["mae_lakhs"],
})
print(comparison)

# overfitting check: error on the rows the model learned from
fit_pred_lakhs = np.expm1(model.predict(X_fit))
fit_report = report_errors(fit_df["Price_Lakhs"], fit_pred_lakhs, fit_df["City"])
print("\nMedian % error, ALL cities | fit rows:", fit_report.loc["ALL", "median_pct_error"],
      "| validation rows:", lgbm_report.loc["ALL", "median_pct_error"])

           baseline_pct  lgbm_pct  baseline_mae  lgbm_mae
City                                                     
Bangalore          22.3      20.1          31.5      29.0
Chennai            19.4      19.4          29.2      24.7
Delhi              19.2      17.9          33.9      32.9
Hyderabad          12.0      13.6          18.7      17.8
Kolkata            39.1      38.1          26.4      25.6
Mumbai             37.5      34.4          58.2      51.3
ALL                26.0      25.1          35.7      32.6

Median % error, ALL cities | fit rows: 22.2 | validation rows: 25.1


# Diagnose before tuning

**Which features does the model use?**

In [15]:
feature_names = model.booster_.feature_name()
importance_gain = model.booster_.feature_importance(importance_type="gain")
importance_split = model.booster_.feature_importance(importance_type="split")

importance_table = pd.DataFrame({
    "gain_%": importance_gain / importance_gain.sum() * 100,
    "splits": importance_split,
}, index=feature_names).round(1)

print(importance_table.sort_values("gain_%", ascending=False))

                        gain_%  splits
Locality                  45.2    1426
Area_SqFt                 40.5    2996
City                       8.4     343
Amenity_Score              4.4     711
BHK                        0.9     300
Resale                     0.7     284
Amenity_Info_Available     0.0       0


In [16]:
def try_settings(extra_settings):
    settings = dict(n_estimators=3000, learning_rate=0.05, num_leaves=31,
                    min_child_samples=20, random_state=RANDOM_STATE, verbose=-1)
    settings.update(extra_settings)

    trial = lgb.LGBMRegressor(**settings)
    trial.fit(X_fit, y_fit, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

    val_errors = report_errors(val_df["Price_Lakhs"], np.expm1(trial.predict(X_val)), val_df["City"])
    fit_errors = report_errors(fit_df["Price_Lakhs"], np.expm1(trial.predict(X_fit)), fit_df["City"])
    return trial, val_errors, fit_errors

**Two diagnostic experiments**

In [17]:
trial_a, val_a, fit_a = try_settings({"min_data_per_group": 10, "max_cat_threshold": 128})
trial_b, val_b, fit_b = try_settings({"num_leaves": 127, "min_child_samples": 5})

summary = pd.DataFrame({
    "baseline": val_baseline_report["median_pct_error"],
    "default": lgbm_report["median_pct_error"],
    "loose_category": val_a["median_pct_error"],
    "big_trees": val_b["median_pct_error"],
})
print(summary)

print("\nTrees used: loose_category", trial_a.best_iteration_, "| big_trees", trial_b.best_iteration_)
print("Fit-row median % error, ALL: default", fit_report.loc["ALL", "median_pct_error"],
      "| loose_category", fit_a.loc["ALL", "median_pct_error"],
      "| big_trees", fit_b.loc["ALL", "median_pct_error"])

           baseline  default  loose_category  big_trees
City                                                   
Bangalore      22.3     20.1            20.6       20.8
Chennai        19.4     19.4            20.6       19.5
Delhi          19.2     17.9            17.4       19.3
Hyderabad      12.0     13.6            13.6       14.1
Kolkata        39.1     38.1            38.0       38.0
Mumbai         37.5     34.4            34.2       34.9
ALL            26.0     25.1            25.0       25.4

Trees used: loose_category 198 | big_trees 82
Fit-row median % error, ALL: default 22.2 | loose_category 22.1 | big_trees 21.5


**checks**

In [18]:
actual = val_df["Price_Lakhs"]
lgbm_val_pred = pd.Series(val_pred_lakhs, index=val_df.index)    # from Cell 14

baseline_error = (val_baseline_pred - actual).abs() / actual * 100
lgbm_error = (lgbm_val_pred - actual).abs() / actual * 100

model_is_closer = lgbm_error < baseline_error
print("Share of flats where LightGBM is closer, ALL:", round(model_is_closer.mean() * 100, 1), "%")
print((model_is_closer.groupby(val_df["City"].astype(str)).mean() * 100).round(1))

Share of flats where LightGBM is closer, ALL: 54.1 %
City
Bangalore    55.1
Chennai      48.9
Delhi        54.6
Hyderabad    52.8
Kolkata      53.8
Mumbai       56.8
dtype: float64


In [19]:
def median_pct_error(actual_values, predicted_values):
    return ((predicted_values - actual_values).abs() / actual_values * 100).median()

long_model = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=127,
                               min_child_samples=5, random_state=RANDOM_STATE, verbose=-1)
long_model.fit(X_fit, y_fit)    # no early stopping this time

for number_of_trees in [50, 100, 200, 400, 800]:
    fit_pred = np.expm1(long_model.predict(X_fit, num_iteration=number_of_trees))
    val_pred = np.expm1(long_model.predict(X_val, num_iteration=number_of_trees))
    print("Trees:", number_of_trees,
          "| fit error:", round(median_pct_error(fit_df["Price_Lakhs"], fit_pred), 1),
          "| validation error:", round(median_pct_error(val_df["Price_Lakhs"], val_pred), 1))

Trees: 50 | fit error: 23.8 | validation error: 26.5
Trees: 100 | fit error: 20.9 | validation error: 25.3
Trees: 200 | fit error: 19.0 | validation error: 25.5
Trees: 400 | fit error: 16.5 | validation error: 25.8
Trees: 800 | fit error: 13.2 | validation error: 25.5


**Helper functions**

In [20]:
def without(columns_to_remove):
    kept = []
    for column in feature_columns:
        if column not in columns_to_remove:
            kept.append(column)
    return kept


def score_feature_set(columns):
    trial_model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=31,
                                    min_child_samples=20, random_state=RANDOM_STATE, verbose=-1)
    trial_model.fit(fit_df[columns], fit_df["Log_Price"],
                    eval_set=[(val_df[columns], val_df["Log_Price"])],
                    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

    predicted_lakhs = np.expm1(trial_model.predict(val_df[columns]))
    errors = report_errors(val_df["Price_Lakhs"], predicted_lakhs, val_df["City"])
    return errors["median_pct_error"], trial_model.best_iteration_

In [21]:
variants = {
    "all7":            feature_columns,
    "no_flag":         without(["Amenity_Info_Available"]),
    "no_flag_resale":  without(["Amenity_Info_Available", "Resale"]),
    "no_flag_amenity": without(["Amenity_Info_Available", "Amenity_Score"]),
    "no_flag_bhk":     without(["Amenity_Info_Available", "BHK"]),
    "core3":           ["City", "Locality", "Area_SqFt"],
}

results = {}
trees_used = {}

for variant_name, columns in variants.items():
    errors, trees = score_feature_set(columns)
    results[variant_name] = errors
    trees_used[variant_name] = trees

print(pd.DataFrame(results))
print()
print("Trees used:", trees_used)

           all7  no_flag  no_flag_resale  no_flag_amenity  no_flag_bhk  core3
City                                                                         
Bangalore  20.1     20.1            20.7             24.0         20.7   23.9
Chennai    19.4     19.4            19.9             23.1         20.1   23.1
Delhi      17.9     17.9            17.8             19.2         18.3   20.3
Hyderabad  13.6     13.6            14.1             14.7         13.7   14.0
Kolkata    38.1     38.1            38.6             38.0         38.3   38.2
Mumbai     34.4     34.4            34.6             36.2         34.7   36.5
ALL        25.1     25.1            25.3             27.1         25.3   27.3

Trees used: {'all7': 202, 'no_flag': 202, 'no_flag_resale': 211, 'no_flag_amenity': 144, 'no_flag_bhk': 195, 'core3': 105}


# Final model Training

In [22]:
final_features = ["City", "Locality", "BHK", "Area_SqFt", "Amenity_Score"]
final_trees = int(202 * 1.1)   # about 10% more than early stopping found

final_model = lgb.LGBMRegressor(
    n_estimators=final_trees, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, random_state=RANDOM_STATE, verbose=-1,
)
final_model.fit(train_df[final_features], train_df["Log_Price"])
print("Trained with", final_trees, "trees")

Trained with 222 trees


**Score the test set**

In [23]:
test_pred_lakhs = pd.Series(np.expm1(final_model.predict(test_df[final_features])),
                            index=test_df.index)
test_report = report_errors(test_df["Price_Lakhs"], test_pred_lakhs, test_df["City"])

final_comparison = pd.DataFrame({
    "baseline_pct": baseline2_report["median_pct_error"],
    "lgbm_pct": test_report["median_pct_error"],
    "baseline_mae": baseline2_report["mae_lakhs"],
    "lgbm_mae": test_report["mae_lakhs"],
    "test_rows": test_report["rows"],
})
print(final_comparison)

           baseline_pct  lgbm_pct  baseline_mae  lgbm_mae  test_rows
City                                                                
Bangalore          24.7      22.4          35.8      32.0       1057
Chennai            16.6      17.6          25.9      24.0        823
Delhi              17.7      16.8          33.8      33.2        746
Hyderabad          11.8      12.2          18.7      16.8        397
Kolkata            38.0      37.1          25.5      24.0       1137
Mumbai             35.9      34.0          57.4      52.9       1273
ALL                25.9      24.7          35.7      33.1       5433


**Price range per city**

In [24]:
ratio = test_df["Price_Lakhs"] / test_pred_lakhs      # actual / predicted
city_labels = test_df["City"].astype(str)

range_table = ratio.groupby(city_labels).quantile([0.1, 0.9]).unstack()
range_table.columns = ["low_ratio", "high_ratio"]
print(range_table.round(2))

low_for_each_row = city_labels.map(range_table["low_ratio"].to_dict())
high_for_each_row = city_labels.map(range_table["high_ratio"].to_dict())
inside = (ratio >= low_for_each_row) & (ratio <= high_for_each_row)
print("Share of test flats inside their city's range:", round(inside.mean() * 100, 1), "%")

           low_ratio  high_ratio
City                            
Bangalore       0.59        1.66
Chennai         0.60        1.61
Delhi           0.62        1.66
Hyderabad       0.77        1.32
Kolkata         0.52        1.95
Mumbai          0.49        2.06
Share of test flats inside their city's range: 79.9 %


In [26]:
artifacts_folder = Path("../artifacts")
artifacts_folder.mkdir(parents=True, exist_ok=True)

# localities available per city, with how many training listings each has
train_locality_counts = train_df["Locality"].astype(str).value_counts().to_dict()
localities_by_city = {}
for city_name in sorted(df["City"].astype(str).unique()):
    names = sorted(df[df["City"].astype(str) == city_name]["Locality"].astype(str).unique())
    localities_by_city[city_name] = [(name, train_locality_counts.get(name, 0)) for name in names]

bundle = {
    "model": final_model,
    "features": final_features,
    "city_categories": list(df["City"].cat.categories),
    "locality_categories": list(df["Locality"].cat.categories),
    "range_ratios": range_table.to_dict(),
    "accuracy_table": test_report.reset_index().to_dict(orient="records"),
    "localities_by_city": localities_by_city,
}



In [27]:
scores = df["Amenity_Score"].dropna()      # use your own dataframe name
print(scores.describe())

bundle["amenity_levels"] = {
    "Not sure": None,
    "Basic": float(scores.quantile(0.25)),
    "Standard": float(scores.quantile(0.50)),
    "Premium": float(scores.quantile(0.80)),
}
print(bundle["amenity_levels"])

count    8191.000000
mean        8.368697
std         6.874938
min         0.000000
25%         2.000000
50%         7.000000
75%        14.000000
max        35.000000
Name: Amenity_Score, dtype: float64
{'Not sure': None, 'Basic': 2.0, 'Standard': 7.0, 'Premium': 15.0}


In [28]:
joblib.dump(bundle, artifacts_folder / "lgbm_model.joblib")
test_report.reset_index().to_csv(artifacts_folder / "accuracy_by_city.csv", index=False)
print("Saved. Model file size (MB):",
      round((artifacts_folder / "lgbm_model.joblib").stat().st_size / 1024**2, 2))

Saved. Model file size (MB): 1.05


**Reload and verify**

In [29]:
bundle = joblib.load(artifacts_folder / "lgbm_model.joblib")

def make_input(city, locality, bhk, area, amenity_score):
    row = pd.DataFrame({
        "City": [city],
        "Locality": [locality],
        "BHK": [bhk],
        "Area_SqFt": [area],
        "Amenity_Score": [amenity_score],
    })
    row["City"] = pd.Categorical(row["City"], categories=bundle["city_categories"])
    row["Locality"] = pd.Categorical(row["Locality"], categories=bundle["locality_categories"])
    return row[bundle["features"]]

sample = test_df.head(5)
rebuilt_pred = []
for _, r in sample.iterrows():
    x = make_input(str(r["City"]), str(r["Locality"]), r["BHK"], r["Area_SqFt"], r["Amenity_Score"])
    rebuilt_pred.append(np.expm1(bundle["model"].predict(x))[0])

print("Rebuilt :", np.round(rebuilt_pred, 1))
print("Original:", test_pred_lakhs.head(5).round(1).values)
print("Match:", np.allclose(rebuilt_pred, test_pred_lakhs.head(5).values))

Rebuilt : [133.6  68.7  51.6  60.1  98.5]
Original: [133.6  68.7  51.6  60.1  98.5]
Match: True
